# Fullstack GPT: #12.0 ~ #12.0

- [x] 새로운 Jupyter notebook에서 리서치 AI 에이전트를 만들고 커스텀 도구를 부여합니다.
- 에이전트는 다음 작업을 수행할 수 있어야 합니다:
    - [x] Wikipedia에서 검색
    - [x] DuckDuckGo에서 검색
    - [x] 웹사이트의 텍스트를 스크랩하고 추출합니다.
    - [x] 리서치 결과를 .txt 파일에 저장하기
- [x] 다음 쿼리로 에이전트를 실행합니다: 
    "Research about the XZ backdoor", the agent should try to search in Wikipedia or DuckDuckGo, if it finds a website in DuckDuckGo it should enter the website and extract it's content, then it should finish by saving the research to a .txt file.

In [3]:
from langchain.globals import set_llm_cache
from langchain.cache import InMemoryCache
from langchain.agents import initialize_agent, AgentType
from langchain.chat_models import ChatOpenAI
from langchain.schema import SystemMessage
from langchain.tools import BaseTool
from langchain.tools.ddg_search import DuckDuckGoSearchRun
from langchain.utilities.wikipedia import WikipediaAPIWrapper
from langchain.utilities.duckduckgo_search import DuckDuckGoSearchAPIWrapper
from langchain.document_loaders import WebBaseLoader
from pydantic import BaseModel, Field
from typing import Type

set_llm_cache(InMemoryCache())

query = "Research about the XZ backdoor"
llm = ChatOpenAI(
    model="gpt-4o-mini-2024-07-18",
    temperature=0.1,
)

class SearchToolArgsSchema(BaseModel):
    query: str = Field(description="The query you will search for")

class WikiSearchTool(BaseTool):
    name="WikiSearchTool"
    description="Use this to search query on Wikipedia."
    args_schema: Type[SearchToolArgsSchema] = SearchToolArgsSchema

    def _run(self, query):
        wiki = WikipediaAPIWrapper()
        return wiki.run(query)

class ScrapeToolArgsSchema(BaseModel):
    url: str = Field(description="The url you'll be accessing and scraping")

class ScrapeTool(BaseTool):
    name="ScrapeTool"
    description="Use this to load contents from the website and parse the html into text."
    args_schema: Type[ScrapeToolArgsSchema] = ScrapeToolArgsSchema

    def _run(self, url):
        loader = WebBaseLoader(url)
        docs = loader.load()
        return "\n\n".join(
            f"Source: {doc.metadata['source']}\n"
            f"title: {doc.metadata['title']}\n"
            f"content: {doc.page_content}"
            for doc in docs
        )

class SaveToolArgsSchema(BaseModel):
    text: str = Field(description="Structured search result from the web that needs to be saved to a txt file.")

class SaveTool(BaseTool):
    name="SaveTool"
    description="Use this to save research to a txt file."
    args_schema: Type[SaveToolArgsSchema] = SaveToolArgsSchema

    def _run(self, text):
        with open("challenge_9.txt", "w", encoding="utf-8") as f:
            f.write(text)
    
ddg_search_tool = DuckDuckGoSearchRun()

agent = initialize_agent(
    llm=llm,
    verbose=True,
    agent=AgentType.OPENAI_FUNCTIONS,
    handle_parsing_errors=True,
    tools=[
        WikiSearchTool(),
        ddg_search_tool,
        ScrapeTool(),
        SaveTool(),
    ],
    agent_kwargs={
        "system_message": SystemMessage(
            content="""
You are a research agent.

When given a topic, follow the steps:
1. Search in Wikipedia or DuckDuckGo
2. If a website is found in DuckDuckGo, enter the website and extract it's content
3. Finish by saving the research to a .txt file
"""
        )
    }
)

prompt = "Research about the XZ backdoor"
agent.invoke(prompt)



> Entering new AgentExecutor chain...

Invoking: `WikiSearchTool` with `{'query': 'XZ backdoor'}`


Page: XZ Utils backdoor
Summary: In February 2024, a malicious backdoor was introduced to the Linux build of the xz utility within the liblzma library in versions 5.6.0 and 5.6.1 by an account using the name "Jia Tan". The backdoor gives an attacker who possesses a specific Ed448 private key remote code execution through OpenSSH (a suite of secure networking utilities) on the affected Linux system. The issue has been given the Common Vulnerabilities and Exposures number CVE-2024-3094 and has been assigned a CVSS score of 10.0, the highest possible score.
While xz is commonly present in most Linux distributions, at the time of discovery the backdoored version had not yet been widely deployed to production systems, but was present in development versions of major distributions. The backdoor was discovered by the software developer Andres Freund, who announced his findings on 29 March 202

{'input': 'Research about the XZ backdoor',
 'output': 'The research on the XZ backdoor has been completed and saved. Here are the key points:\n\n- In February 2024, a malicious backdoor was introduced to the Linux build of the xz utility within the liblzma library in versions 5.6.0 and 5.6.1 by an account using the name "Jia Tan". \n- The backdoor allows an attacker with a specific Ed448 private key to execute remote code through OpenSSH on affected Linux systems. \n- This vulnerability is identified as CVE-2024-3094 and has a CVSS score of 10.0, the highest possible score.\n- The backdoored version was not widely deployed in production systems at the time of discovery but was present in development versions of major distributions.\n- The backdoor was discovered by software developer Andres Freund, who announced his findings on March 29, 2024.\n\nIf you need further information or details, feel free to ask!'}